# RNNs & LSTMs in Vision
Today we will cover:
- Theory: Sequence modeling for video
- Exercise: Action recognition with CNN + LSTM
- Mini Project: Gesture recognition

## Theory: Sequence Modeling for Video

- Video data = sequence of frames.
- CNNs extract spatial features (objects, shapes, textures).
- RNNs/LSTMs capture temporal dependencies (motion, order).
- Applications: action recognition, gesture recognition, video captioning.

### LSTMs
- Long Short-Term Memory networks handle long-range dependencies.
- Gates: input, forget, output.
- Useful for variable-length sequences.

### CNN + LSTM Pipeline
- CNN: feature extractor per frame.
- LSTM: sequence model across frames.
- Output: action/gesture class.

## Setup: Video Dataset Loader (Action Recognition)

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import cv2
import os

# Example dataset class (replace with actual video dataset path)
class VideoDataset(Dataset):
    def __init__(self, video_dir, transform=None, seq_len=16):
        self.video_dir = video_dir
        self.videos = os.listdir(video_dir)
        self.transform = transform
        self.seq_len = seq_len
    def __len__(self):
        return len(self.videos)
    def __getitem__(self, idx):
        video_path = os.path.join(self.video_dir, self.videos[idx])
        cap = cv2.VideoCapture(video_path)
        frames = []
        while len(frames) < self.seq_len and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if self.transform:
                frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        frames = torch.stack(frames)
        label = 0  # placeholder, replace with actual label
        return frames, label

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((64,64))
])

train_dataset = VideoDataset('./videos/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
print("Loaded video dataset with", len(train_dataset), "videos")

## Exercise: Action Recognition with CNN + LSTM

In [ ]:
# Define CNN + LSTM model
class CNN_LSTM(nn.Module):
    def __init__(self, hidden_dim=128, num_classes=5):
        super(CNN_LSTM, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.lstm = nn.LSTM(32*16*16, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        batch_size, seq_len, c, h, w = x.size()
        cnn_out = []
        for t in range(seq_len):
            out = self.cnn(x[:,t])
            out = out.view(batch_size, -1)
            cnn_out.append(out)
        cnn_out = torch.stack(cnn_out, dim=1)
        lstm_out, _ = self.lstm(cnn_out)
        out = self.fc(lstm_out[:,-1,:])
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN_LSTM().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop (simplified)
for epoch in range(1, 3):
    running_loss = 0.0
    for frames, labels in train_loader:
        frames, labels = frames.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}, Loss: {running_loss/len(train_loader):.4f}")

## Mini Project: Gesture Recognition Prototype

In [ ]:
import matplotlib.pyplot as plt

# Evaluate on a batch
frames, labels = next(iter(train_loader))
outputs = model(frames.to(device))
_, predicted = torch.max(outputs, 1)

print("Predicted gesture class:", predicted[0].item())

# Display first frame of sequence
plt.imshow(frames[0][0].permute(1,2,0))
plt.title("Sample Gesture Frame")
plt.show()